# 14 — Full-event evaluation: autoregressive vs teacher-forced

Two ways to score next-event prediction, for the **naive baseline** and the
**disjoint-heads model (v5)**, on the val split:

- **teacher-forced** — when predicting field *k*, the true values of fields
  *0…k-1* of the same event are revealed. (This is what the v5 evaluation
  headline reports.)
- **autoregressive** — the model predicts field 0, feeds *its own prediction*
  back in, predicts field 1, and so on through `result`. Only the 3-event
  history and the always-on match state (`minute`, `second`, goal/card counts)
  are true; every current-event field is the model's own guess.

The gap between the two columns is the cost of error compounding down the
7-field chain. The naive baseline ignores context entirely, so its two columns
are identical by construction — it's a flat reference in both.

Fields predicted by argmax (greedy), every event in val including boundary
events, same metric definitions as the v5 headline (accuracy for
`is_home`/`event`/`result`, MAE otherwise).

In [1]:
import json

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F

train = pd.read_csv("data/train_set.csv")
val = pd.read_csv("data/val_set.csv")

event_cols = ["is_home", "event", "start_x", "start_y", "time_elapsed", "duration", "result"]
categorical = ["is_home", "event", "result"]
continuous_fields = ["start_x", "start_y", "time_elapsed", "duration"]
match_state_fields = [
    "period_id", "minute", "second",
    "home_goal_count", "away_goal_count",
    "home_red_count", "away_red_count",
    "home_yellow_count", "away_yellow_count",
]
embed_fields = ["is_home", "event", "result"]


def with_lags(fields, n_lags=3):
    cols = list(fields)
    for lag in range(1, n_lags + 1):
        cols += [f"{f}-{lag}" for f in fields]
    return cols


continuous_cols = with_lags(continuous_fields) + match_state_fields
embed_cols = {f: [f] + [f"{f}-{lag}" for lag in range(1, 4)] for f in embed_fields}

with open("data/event_masks.json") as f:
    raw_masks = json.load(f)
with open("data/category_lookup.json") as f:
    lookup = {c: {int(k): v for k, v in m.items()} for c, m in json.load(f).items()}

## Shared preprocessing (matches v5 training exactly)

In [2]:
def remap_sentinel(values):
    values = np.asarray(values, dtype=float)
    values = np.where(values == 106, -2.0, values)
    return (values + 2).astype(np.int64)


def normalize_field(values, mean, std):
    values = np.asarray(values, dtype=float)
    is_sentinel = (values == -1) | (values == -2)
    return np.where(is_sentinel, values, (values - mean) / std + 10)


# stats from train only, sentinels excluded — same as the v5 notebook
continuous_stats = {}
for field in continuous_fields:
    cols = [field] + [f"{field}-{lag}" for lag in range(1, 4)]
    v = train[cols].values.flatten().astype(float)
    v = v[(v != -1) & (v != -2)]
    continuous_stats[field] = (v.mean(), v.std())
for field in ["minute", "second"]:
    v = train[field].values.astype(float)
    v = v[(v != -1) & (v != -2)]
    continuous_stats[field] = (v.mean(), v.std())

masks = {c: remap_sentinel(v).tolist() for c, v in raw_masks.items()}
shifted_of = {f: torch.tensor(masks[f]) for f in event_cols}          # local idx -> shifted code
local_of = {}
for f in event_cols:
    pl = torch.full((108,), -1, dtype=torch.long)
    pl[torch.tensor(masks[f])] = torch.arange(len(masks[f]))
    local_of[f] = pl                                                  # shifted code -> local idx

In [3]:
def build_cont(df):
    X = df[continuous_cols].values.astype(float).copy()
    for field in continuous_fields:
        m, s = continuous_stats[field]
        for col in [field] + [f"{field}-{lag}" for lag in range(1, 4)]:
            j = continuous_cols.index(col)
            X[:, j] = normalize_field(X[:, j], m, s)
    for field in ["minute", "second"]:
        m, s = continuous_stats[field]
        j = continuous_cols.index(field)
        X[:, j] = normalize_field(X[:, j], m, s)
    return torch.tensor(X).float()


def build_embed(df):
    return {f: torch.tensor(remap_sentinel(df[c].values)).long() for f, c in embed_cols.items()}


def field_targets(df, field):
    return df.loc[df["step"] == event_cols.index(field), "target"].to_numpy()

## The disjoint-heads model (v5)

In [4]:
ckpt = torch.load("data/mlp_embeddings_disjoint_checkpoint.pt", weights_only=False)
C = {"event": ckpt["C_event"], "is_home": ckpt["C_is_home"], "result": ckpt["C_result"]}
W1, b1, W2, b2, W3, b3 = ckpt["W1"], ckpt["b1"], ckpt["W2"], ckpt["b2"], ckpt["W3"], ckpt["b3"]
output_heads = ckpt["output_heads"]


def trunk(X_cont, X_embed):
    embs = [C[k][X_embed[k]].reshape(X_embed[k].shape[0], -1) for k in embed_fields]
    h = torch.cat([X_cont, *embs], dim=1)
    h = torch.relu(h @ W1 + b1)
    h = torch.relu(h @ W2 + b2)
    h = torch.relu(h @ W3 + b3)
    return h


@torch.no_grad()
def v5_predict_field(X_cont, X_embed, field):
    logits = trunk(X_cont, X_embed) @ output_heads[field][0] + output_heads[field][1]
    local = logits.argmax(1)
    return (shifted_of[field][local] - 2).numpy()            # -> raw space (-2 boundary, 0.. real)

### Teacher-forced

Each `step == k` row already has fields `0…k-1` filled with their true values
and `k…6` masked — exactly the teacher-forced input for predicting field `k`.

In [5]:
def v5_teacher_forced():
    out = {}
    for field in event_cols:
        rows = val[val["step"] == event_cols.index(field)]
        pred = v5_predict_field(build_cont(rows), build_embed(rows), field)
        out[field] = pred
    return out

### Autoregressive

Start from every event's `step == 0` row (all current fields masked), then walk
`is_home → … → result`, writing each argmax prediction back into the input
before predicting the next field.

In [6]:
def v5_autoregressive():
    start = val[val["step"] == 0].copy()
    X_cont = build_cont(start)
    X_embed = build_embed(start)

    preds = {}
    for field in event_cols:
        pred_raw = v5_predict_field(X_cont, X_embed, field)
        preds[field] = pred_raw

        shifted = torch.tensor(remap_sentinel(pred_raw))
        if field in embed_fields:
            X_embed[field] = X_embed[field].clone()
            X_embed[field][:, 0] = shifted
        else:                                                # continuous current field
            m, s = continuous_stats[field]
            X_cont = X_cont.clone()
            X_cont[:, continuous_fields.index(field)] = torch.tensor(
                normalize_field(pred_raw, m, s)
            ).float()
    return preds

## The naive baseline (mode / median, from train)

In [7]:
baseline_pred = {}
for field in event_cols:
    tr = field_targets(train, field)
    if field in categorical:
        vals, counts = np.unique(tr, return_counts=True)
        baseline_pred[field] = int(vals[counts.argmax()])
    else:
        baseline_pred[field] = float(np.median(tr))
baseline_pred

{'is_home': 1,
 'event': 1,
 'start_x': 54.0,
 'start_y': 34.0,
 'time_elapsed': 1.0,
 'duration': 1.0,
 'result': 0}

## Score everything into one table

In [8]:
def score(pred_raw, field):
    true_raw = field_targets(val, field)                     # 106 -> boundary; matches remap below
    true_raw = np.where(true_raw == 106, -2, true_raw)
    p = np.where(np.asarray(pred_raw) == 106, -2, np.asarray(pred_raw))
    if field in categorical:
        return ("acc", float(np.mean(p == true_raw)))
    return ("mae", float(np.mean(np.abs(p - true_raw))))


v5_tf = v5_teacher_forced()
v5_ar = v5_autoregressive()

records = []
for field in event_cols:
    metric, base_val = score(np.full(len(field_targets(val, field)), baseline_pred[field]), field)
    _, v5_tf_val = score(v5_tf[field], field)
    _, v5_ar_val = score(v5_ar[field], field)
    # "cost" = how much worse autoregressive is: higher-is-better for acc,
    # lower-is-better for mae, so flip the sign for acc.
    cost = (v5_tf_val - v5_ar_val) if metric == "acc" else (v5_ar_val - v5_tf_val)
    records.append({
        "field": field,
        "metric": metric,
        "baseline (both)": round(base_val, 3),
        "v5 teacher-forced": round(v5_tf_val, 3),
        "v5 autoregressive": round(v5_ar_val, 3),
        "v5 compounding cost": round(cost, 3),
    })

table = pd.DataFrame(records)
table

,field,metric,baseline (both),v5 teacher-forced,v5 autoregressive,v5 compounding cost
0,is_home,acc,0.518,0.950,0.950,0.000
1,event,acc,0.430,0.755,0.737,0.018
2,start_x,mae,20.634,6.505,6.748,0.243
3,start_y,mae,17.542,6.894,7.444,0.551
4,time_elapsed,mae,2.192,0.913,1.425,0.512
5,duration,mae,0.899,0.753,0.887,0.134
6,result,acc,0.827,0.884,0.822,0.063


### Read-out

- `baseline (both)` is one column because the mode/median predictor never looks
  at earlier fields — autoregressive and teacher-forced are identical for it.
- `v5 compounding cost` is normalized so **positive always means autoregressive
  is worse** (accuracy dropped, or MAE grew) — it's the price of the model
  leaning on its own earlier guesses instead of the truth.
